In [0]:
%skip
file_path = '/Volumes/yatindra_org/microfin/dataset_volume/autoloader_files/data_files/'
schema_location = '/Volumes/yatindra_org/microfin/dataset_volume/autoloader_files/autoloader_schema/'
checkpoint_location = '/Volumes/yatindra_org/microfin/dataset_volume/autoloader_files/checkpoint_location/'

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path)
)

(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .trigger(processingTime="1 minute")
    .toTable("yatindra_org.microfin.orders_stream")
)


In [0]:
# As serverless compute does not support streaming live, hence I have to run this load as batch style.

file_path = '/Volumes/yatindra_org/microfin/dataset_volume/autoloader_files/data_files/'
schema_location = '/Volumes/yatindra_org/microfin/dataset_volume/autoloader_files/autoloader_schema/'
checkpoint_location = '/Volumes/yatindra_org/microfin/dataset_volume/autoloader_files/checkpoint_location/'
schema = 'OrderID int, OrderDate string, CustomerID int, TotalAmount string, Status string'   # DDL-style schema string

# Read stream with explicit schema
df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("rescuedDataColumn", "_rescued_data")
        .schema(schema)
        .option("header", "true")
        .load(file_path)
)

# Write stream in batch mode
(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable("yatindra_org.microfin.orders_stream")
)


In [0]:
%sql
select * from yatindra_org.microfin.orders_stream;
-- truncate table yatindra_org.microfin.orders_stream;
-- DROP TABLE IF EXISTS yatindra_org.microfin.orders_stream;